# K2 (mock feed) — the update path against a server of its own

**What this is:** the same update checks as `updater`, but against a **mock GitHub on 127.0.0.1** and in a **data folder
of its own**, so nothing touches your installed app, its config, or the internet. It needs an exe built with the test
flags (`--features test-env`): a local build (`npx --yes @tauri-apps/cli@2 build --no-bundle --features test-env`, the exe is
`target\release\resonance-stream.exe`) or a release candidate. The released 0.6.0 / 0.6.1 ignore the flags.

**How it works:** the notebook serves a feed announcing a "new version" (this same exe with a few bytes appended, signed with
**your backup key**), starts a fresh copy of the exe per check with `--data-dir ... --feed-url ... --status-file ...`, and
reads what happened from the **status file** (`update`: `available:<v>` / `downloading` / `downloaded` / `error:<reason>`)
and from the copy's folder (`.old`, `update_temp.exe`, the exe's hash). You click in the app only where the notebook says `>>>`.

**Before you start:** close the installed app and any other copy. Run Jupyter as Administrator. Have the backup key file + its password.
**Every check closes its app by itself** when it ends -- also when it fails or raises an error (a broken check is a failed row and
the notebook goes on) -- and first closes any copy an earlier check left running.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import common, mockfeed, updater
from runbook.common import Recorder, capture, ask_text
import getpass, json, os, shutil, textwrap, time, types
rec = Recorder("updater-mock")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
RUNS = common.RUNS / "mock"
# Your real config must come out of this notebook exactly as it went in.
REAL_CONFIG = updater.app_config_path()
REAL_BEFORE = mockfeed.fingerprint(REAL_CONFIG)
print("real config:", REAL_CONFIG, "(exists)" if REAL_BEFORE else "(missing)")

## 0 · The exe under test
It must understand the test flags. The check runs it with `--print-env` (it writes its status file and exits at once);
an exe that ignores the flags would open its window instead and is stopped after 30 seconds.

In [ ]:
LOCAL_EXE = Path(ask_text("LOCAL_EXE", "Full path of the test-env exe (e.g. C:/.../target/release/resonance-stream.exe), then Enter: "))
if not LOCAL_EXE.is_file():
    raise SystemExit(f"{LOCAL_EXE} is not a file")
smoke = RUNS / "smoke"
shutil.rmtree(smoke, ignore_errors=True)
smoke.mkdir(parents=True)
code_, out = mockfeed.run_print_env(LOCAL_EXE, smoke / "data", smoke / "status.json", timeout=30)
st = mockfeed.read_status(smoke / "status.json")
understood = code_ == 0 and st is not None and "print-env" in st.get("flags", [])
rec.auto("M0", "the exe understands the test flags (--print-env wrote its status file)", understood,
         f"version {st['version']}, data_dir {st['data_dir']}" if understood else " ".join(out.split())[-200:] or f"exit {code_}, no status file")
if not understood:
    raise SystemExit("This exe does not take the test flags -- build it with --features test-env.")
LOCAL_VERSION = st["version"]
OLD_SHA = updater.sha256_file(LOCAL_EXE)
print("exe under test:", LOCAL_EXE, "| version", LOCAL_VERSION, "| sha256", OLD_SHA[:16])

## 1 · A "new version", signed with the backup key
The new exe is the exe under test with a marker appended (Windows ignores bytes after the last section, so it still starts,
and its hash differs). The key's contents go in `TAURI_SIGNING_PRIVATE_KEY` for the signing only, as in `updater` A5; the
password is typed into a hidden prompt and never printed or saved. Needs Node (`npx`).

In [ ]:
NEW_VERSION = ask_text("NEW_VERSION", "Version the mock release announces [9.9.9]: ") or "9.9.9"
NEW_EXE = mockfeed.make_new_exe(LOCAL_EXE, RUNS / "new" / "update.exe")
NEW_SHA = updater.sha256_file(NEW_EXE)
key_path = ask_text("BACKUP_KEY", "Full path of the BACKUP key file (e.g. C:/keys/backup.key), then Enter: ")
password = os.environ.get("RUNBOOK_TEXT_BACKUP_PW", "") if common.dry_run() else getpass.getpass("Backup key password: ")
sig = Path(str(NEW_EXE) + ".sig")
sig.unlink(missing_ok=True)
SIGNATURE = None
try:
    key_text = "dry-run-key" if (common.dry_run() and key_path) else updater.read_key_file(key_path)
except ValueError as e:
    key_text = None
    rec.auto("M1-1", "the backup key file can be read", False, str(e))
if key_text is not None:
    os.environ["TAURI_SIGNING_PRIVATE_KEY"] = key_text
    os.environ["TAURI_SIGNING_PRIVATE_KEY_PASSWORD"] = password
    try:
        code_, out = capture(["npx", "--yes", "@tauri-apps/cli@2", "signer", "sign", "--app-version", NEW_VERSION, str(NEW_EXE)],
                             fixture="tauri_sign_ok.txt", timeout=900)
    finally:
        os.environ.pop("TAURI_SIGNING_PRIVATE_KEY", None)
        os.environ.pop("TAURI_SIGNING_PRIVATE_KEY_PASSWORD", None)
        key_text = password = ""
    if common.dry_run():
        shutil.copy(common.FIXTURES / "fake_backup.sig", sig)
    signed = code_ == 0 and sig.exists()
    rec.auto("M1-1", "the new exe was signed", signed,
             "signature written" if signed else " ".join(out.split())[-200:])
    if signed:
        sig_text = sig.read_text()
        key_id = updater.signature_key_id(sig_text)
        rec.auto("M1-2", "...by the BACKUP key", updater.is_backup_key(key_id), f"key id {key_id}")
        rec.auto("M1-3", f"...for version {NEW_VERSION}", updater.signature_version(sig_text) == NEW_VERSION,
                 f"the signature says {updater.signature_version(sig_text)}")
        SIGNATURE = mockfeed.feed_signature(sig_text)
if SIGNATURE is None:
    raise SystemExit("Cannot go on without a signed new exe (see the M1 rows above).")
print("new exe:", NEW_EXE, "| sha256", NEW_SHA[:16])

## Helpers
Each check starts **a fresh copy** of the exe in a folder of its own (an update replaces the exe, so a copy is single-use),
with its own data folder, status file and log file under `runs/mock/<check>-N/`.

In [ ]:
def start_app(label, server):
    exe = updater.fresh_copy(LOCAL_EXE, RUNS, label)
    app = types.SimpleNamespace(label=label, exe=exe, dir=exe.parent, status=exe.parent / "status.json",
                                log=exe.parent / "app.log", pid=None)
    args = mockfeed.flag_args(app.dir / "data", app.status, log_file=app.log,
                              feed_url=server.feed_url, metadata_url=server.metadata_url)
    app.pid = mockfeed.start_app(exe, args)
    return app

def current(app):
    return mockfeed.read_status(app.status) or {}

def wait(app, predicate, timeout):
    return mockfeed.wait_status(app.status, predicate, timeout)

def kind_in(*kinds):
    return lambda s: mockfeed.update_kind(s.get("update"))[0] in kinds

def await_update(app, kinds, timeout):
    st = wait(app, kind_in(*kinds), timeout)
    return st, (mockfeed.update_kind(st.get("update")) if st else ("timeout", ""))

def diagnose(app):
    print(f"  [{app.label}] status file: {json.dumps(current(app), ensure_ascii=False)}")
    tail = mockfeed.log_tail(app.log)
    print("  app log, last lines:\n" + textwrap.indent(tail, "    ") if tail else "  (the app wrote no log)")

def guard(check):
    # One check as try / catch / finally: an exception is a failed row `<check>-error` (traceback printed) and the
    # notebook goes on; whatever happens, every copy under runs/mock is closed afterwards -- the restarted one too --
    # and copies an earlier check left running are closed first.
    return updater.step(rec, check, RUNS, strays=RUNS, stop=mockfeed.stop_copies)

def begin(label, server):
    app = start_app(label, server)
    st = wait(app, lambda s: s.get("ready"), 120)
    rec.auto(label, "the copy started with the test flags and became ready", st is not None,
             f"pid {st['pid']}, data in {st['data_dir']}" if st else "the status file never said ready")
    if st is None:
        diagnose(app)
    return app, st

def new_server(**kw):
    return mockfeed.MockServer(exe_bytes=kw.pop("exe_bytes", NEW_EXE.read_bytes()), signature=SIGNATURE,
                               version=kw.pop("version", NEW_VERSION), **kw)

def announced(label, app):
    st, (kind, detail) = await_update(app, ("available", "error"), 90)
    ok = kind == "available" and detail == NEW_VERSION
    rec.auto(f"{label}-1", "the feed's newer version is announced (update: available)", ok,
             f"update: {st.get('update') if st else 'no answer in 90 s'}")
    if not ok:
        diagnose(app)
    return ok

## 2 · What the app does with a feed (no clicks)
Each case serves one kind of feed and reads the status file's `update`. For "nothing new" cases the notebook waits until the
app has asked for the feed and then 3 seconds (the status file has no "check finished" marker). **M12** has no server at all.

In [ ]:
def feed_case(label, title, expect, *, feed_mode="ok", version=None, down=False):
    with guard(label), new_server(version=version or NEW_VERSION) as server:
        server.feed_mode = feed_mode
        if down:
            server.stop()          # the URLs stay, nothing answers
        app, st = begin(label + "-app", server)
        if expect == "error":
            st, (kind, detail) = await_update(app, ("error",), 60)
        else:
            if down:
                time.sleep(3 if common.dry_run() else 20)
            else:
                for _ in range(120):
                    if any(h.endswith("/latest.json") for h in server.hits):
                        break
                    time.sleep(0.5)
                time.sleep(3)
            st = current(app)
            kind, detail = mockfeed.update_kind(st.get("update"))
        rec.auto(label, title, kind == expect, f"update: {st.get('update') if st else 'no status'}")
        if kind != expect:
            diagnose(app)

feed_case("M8", "a feed that is not JSON ends in update: error", "error", feed_mode="garbage")
feed_case("M9", "a feed announcing the version already running is no update", "none", version=LOCAL_VERSION)
feed_case("M10", "a feed announcing an OLDER version is no update", "none", version="0.0.1")
feed_case("M11", "a feed that is a 404 is no update (no stable release yet)", "none", feed_mode="404")
feed_case("M12", "nothing answering at all is no update (and no crash)", "none", down=True)

## 3 · A good update, end to end
Watch the dialog: Korean notes? the bar moving? **다운로드 완료** only after the bar is full? The notebook waits for each step
in the status file and tells you what to press (`>>>`).

In [ ]:
with guard("M3"), new_server() as server:
    app, st = begin("M3", server)
    old_pid = (st or {}).get("pid")
    kind = ""
    if st and announced("M3", app):
        print("\n>>> In the app: the update dialog is open. Press the button that STARTS the update. (waiting up to 5 minutes)")
        st, (kind, detail) = await_update(app, ("downloaded", "error"), 300)
        rec.auto("M3-2", "the download finishes and passes the signature check (update: downloaded)", kind == "downloaded",
                 f"update: {st.get('update') if st else 'nothing in 5 minutes'}")
        if kind != "downloaded":
            diagnose(app)
    if kind == "downloaded":
        print("\n>>> Now press 재시작. (waiting up to 3 minutes)")
        st2 = wait(app, lambda s: s.get("ready") and s.get("pid") != old_pid, 180)
        rec.auto("M3-3", "the app restarted into a new process, with the same flags", st2 is not None,
                 f"pid {old_pid} -> {st2.get('pid')}" if st2 else "no new process reported ready")
        if st2:
            rec.auto("M3-4", "the new process runs the swapped exe (same path)", Path(st2["exe"]).resolve() == app.exe.resolve(),
                     st2["exe"])
        for title, ok, evidence in updater.check_a1(app.dir, app.exe.name, OLD_SHA, NEW_SHA):
            rec.auto("M3-5", title, ok, evidence)
        if st2 is None:
            diagnose(app)
    rec.manual("M3-ui", "the dialog behaved",
               "Think back over the update you just did.\n"
               "Korean release notes in the dialog?  The bar moved?  '다운로드 완료' only AFTER the bar was full and a moment of checking?  "
               "After 재시작 a new window with the app?",
               "all of that")

## 4 · A tampered download is refused
The notebook waits for **다운로드 완료**, flips one byte in `update_temp.exe`, and asks you to press 재시작.

In [ ]:
with guard("M4"), new_server() as server:
    app, st = begin("M4", server)
    old_pid = (st or {}).get("pid")
    kind = ""
    if st and announced("M4", app):
        print("\n>>> Press the button that STARTS the update; when it says 다운로드 완료, wait. (waiting up to 5 minutes)")
        st, (kind, detail) = await_update(app, ("downloaded", "error"), 300)
        rec.auto("M4-2", "the download finishes (update: downloaded)", kind == "downloaded",
                 f"update: {st.get('update') if st else 'nothing in 5 minutes'}")
    temp = app.dir / "update_temp.exe"
    if kind == "downloaded" and temp.exists():
        offset, before, after = updater.flip_one_byte(temp)
        rec.auto("M4-3", "update_temp.exe: one byte flipped", True, f"offset {offset}: {before:#04x} -> {after:#04x}")
        print("\n>>> NOW press 재시작. It must be REFUSED: an error in the dialog, the app keeps running the old version.")
        rec.manual("M4-ui", "재시작 on the tampered file is refused",
                   "Press 재시작 and read what the dialog says.",
                   "an error step (the file is not signed by a trusted key); the app does NOT restart")
        time.sleep(3)
        rec.auto("M4-4", "no new process started", current(app).get("pid") == old_pid,
                 f"pid {old_pid} -> {current(app).get('pid')}")
        for title, ok, evidence in updater.check_a2(app.dir, app.exe.name, OLD_SHA):
            rec.auto("M4-5", title, ok, evidence)
    elif kind == "downloaded":
        rec.auto("M4-3", "update_temp.exe is next to the exe", False, f"not found in {app.dir}")
        diagnose(app)

## 5 · A download that fails: bad signature, cut connection, silence
Three servers that go wrong on purpose. Each time press the button that **starts** the update, then wait: the status file
must say `update: error`, and no half file may be called `update_temp.exe`. The silent one waits for the app's 30-second
no-data limit (K1: on 0.6.0 this hung at 0%).

In [ ]:
def failing_download(label, title, exe_mode=None, exe_bytes=None, timeout=120):
    kw = {} if exe_bytes is None else {"exe_bytes": exe_bytes}
    with guard(label), new_server(**kw) as server:
        if exe_mode:
            server.exe_mode = exe_mode
        app, st = begin(label, server)
        if st and announced(label, app):
            print(f"\n>>> {label}: press the button that STARTS the update. (waiting up to {timeout // 60 + 1} minutes)")
            started = time.monotonic()
            st, (kind, detail) = await_update(app, ("error", "downloaded"), timeout + 60)
            took = time.monotonic() - started
            rec.auto(f"{label}-2", title, kind == "error",
                     f"update: {st.get('update') if st else 'no answer'} (after {took:.0f} s incl. your click)")
            left = updater.leftovers(app.dir, app.exe.name)
            rec.auto(f"{label}-3", "no half file named update_temp.exe is left", not left["temp"] and not left["part"],
                     "update_temp.exe exists" if left["temp"] else (".part file left" if left["part"] else ""))
            if kind != "error":
                diagnose(app)

flipped = bytearray(NEW_EXE.read_bytes())
flipped[len(flipped) // 2] ^= 0xFF
failing_download("M5", "a download that does not match its signature is refused (update: error)", exe_bytes=bytes(flipped))
failing_download("M6", "a connection cut mid-download ends in update: error", exe_mode="cut")
failing_download("M7", "a download that goes silent ends in update: error within the stall limit", exe_mode="stall", timeout=150)

## Your real data must be untouched

In [ ]:
mockfeed.stop_copies(RUNS)     # nothing may be left running
after = mockfeed.fingerprint(REAL_CONFIG)
rec.auto("M-iso", "your real config.json was not touched by any of this", after == REAL_BEFORE,
         "same size, time and hash as before" if after == REAL_BEFORE else f"before {REAL_BEFORE}, after {after}")

## Report
Paste the output of the next cell to Claude. The copies and their logs stay under `runs/mock/` (ignored by git).

In [ ]:
print(rec.report())